# CMI Sensor Behavior Replica — Starter Notebook
competition Link: https://www.kaggle.com/t/2024e94257d04500a0697271eefb8864

This is a clean starter notebook for classying sensor behavior competition.

Each `sequence_id` is a time series recorded from a wrist-worn device.  
The task is to predict one `gesture` label for every sequence in `test.csv`.

This notebook intentionally keeps things simple:

- Normal Kaggle submission: `sequence_id,gesture`
- No API inference server
- No world-coordinate acceleration transformation   
- No special rotation missing-value handling
- Simple sequence aggregation features
- LightGBM baseline if available, otherwise sklearn fallback

The local score uses the same CMI-style hierarchical F1 idea:
1. binary F1 for target vs non-target behavior
2. macro F1 where all non-target gestures are collapsed into one class

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option("display.max_columns", 200)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

print("LightGBM available:", HAS_LGBM)

LightGBM available: True


## 1. Load data

Change `INPUT_DIR` to match your private Kaggle competition dataset path.

Expected files:

- `train.csv`
- `test.csv`
- `train_demographics.csv`
- `test_demographics.csv`
- `sample_submission.csv`

In [2]:
INPUT_DIR = Path("/kaggle/input/competitions/comp-sensor-timeseries")

train = pd.read_csv(INPUT_DIR / "train.csv")
test = pd.read_csv(INPUT_DIR / "test.csv")
train_demo = pd.read_csv(INPUT_DIR / "train_demographics.csv")
test_demo = pd.read_csv(INPUT_DIR / "test_demographics.csv")
sample_submission = pd.read_csv(INPUT_DIR / "sample_submission.csv")

print("train:", train.shape)
print("test:", test.shape)
print("train_demo:", train_demo.shape)
print("test_demo:", test_demo.shape)
print("sample_submission:", sample_submission.shape)

display(train.head())
display(sample_submission.head())

train: (459122, 16)
test: (115823, 15)
train_demo: (65, 8)
test_demo: (16, 8)
sample_submission: (1621, 2)


,row_id,sequence_type,sequence_id,sequence_counter,subject,orientation,behavior,phase,gesture,acc_x,acc_y,acc_z,rot_w,rot_x,rot_y,rot_z
0,SEQ_000007_000000,Target,SEQ_000007,0,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.683594,6.214844,3.355469,0.134399,-0.355164,-0.447327,-0.809753
1,SEQ_000007_000001,Target,SEQ_000007,1,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.949219,6.214844,3.125000,0.143494,-0.340271,-0.428650,-0.824524
2,SEQ_000007_000002,Target,SEQ_000007,2,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.722656,5.410156,5.421875,0.219055,-0.274231,-0.356934,-0.865662
3,SEQ_000007_000003,Target,SEQ_000007,3,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,6.601562,3.531250,6.457031,0.297546,-0.264160,-0.238159,-0.885986
4,SEQ_000007_000004,Target,SEQ_000007,4,SUBJ_059520,Seated Lean Non Dom - FACE DOWN,Relaxes and moves hand to target location,Transition,Cheek - pinch skin,5.566406,0.277344,9.632812,0.333557,-0.218628,-0.063538,-0.914856


,sequence_id,gesture
0,SEQ_000016,Above ear - pull hair
1,SEQ_000018,Above ear - pull hair
2,SEQ_000166,Above ear - pull hair
3,SEQ_000169,Above ear - pull hair
4,SEQ_000174,Above ear - pull hair


## 2. Local metric

In [3]:
TARGET_GESTURES = [
    "Above ear - pull hair",
    "Cheek - pinch skin",
    "Eyebrow - pull hair",
    "Eyelash - pull hair",
    "Forehead - pull hairline",
    "Forehead - scratch",
    "Neck - pinch skin",
    "Neck - scratch",
]

def hierarchical_f1_score(y_true, y_pred):
    y_true = pd.Series(y_true)
    y_pred = pd.Series(y_pred)

    y_true_binary = y_true.isin(TARGET_GESTURES)
    y_pred_binary = y_pred.isin(TARGET_GESTURES)

    f1_binary = f1_score(
        y_true_binary,
        y_pred_binary,
        pos_label=True,
        average="binary",
        zero_division=0,
    )

    y_true_mc = y_true.apply(lambda x: x if x in TARGET_GESTURES else "non_target")
    y_pred_mc = y_pred.apply(lambda x: x if x in TARGET_GESTURES else "non_target")

    f1_macro = f1_score(
        y_true_mc,
        y_pred_mc,
        average="macro",
        zero_division=0,
    )

    return 0.5 * f1_binary + 0.5 * f1_macro

## 3. Sequence-level features

The raw file has many rows per sequence.  
We aggregate each sequence into one row.

Starter features:

- mean
- standard deviation
- minimum
- maximum
- median
- first value
- last value
- last minus first
- sequence length

We use only numeric sensor columns from this simplified dataset:

- `acc_x`, `acc_y`, `acc_z`
- `rot_w`, `rot_x`, `rot_y`, `rot_z`

In [4]:
ID_COL = "sequence_id"
TARGET_COL = "gesture"
SUBJECT_COL = "subject"

SENSOR_COLS = [
    "acc_x", "acc_y", "acc_z",
    "rot_w", "rot_x", "rot_y", "rot_z",
]

SENSOR_COLS = [c for c in SENSOR_COLS if c in train.columns and c in test.columns]
print("Using sensor columns:", SENSOR_COLS)

def make_sequence_features(df):
    grouped = df.groupby(ID_COL, sort=False)

    pieces = []

    stats = grouped[SENSOR_COLS].agg(["mean", "std", "min", "max", "median"])
    stats.columns = [f"{col}_{stat}" for col, stat in stats.columns]
    pieces.append(stats)

    first_values = grouped[SENSOR_COLS].first()
    first_values.columns = [f"{c}_first" for c in first_values.columns]
    pieces.append(first_values)

    last_values = grouped[SENSOR_COLS].last()
    last_values.columns = [f"{c}_last" for c in last_values.columns]
    pieces.append(last_values)

    delta_values = grouped[SENSOR_COLS].last() - grouped[SENSOR_COLS].first()
    delta_values.columns = [f"{c}_delta" for c in delta_values.columns]
    pieces.append(delta_values)

    seq_len = grouped.size().to_frame("sequence_length")
    pieces.append(seq_len)

    # Keep subject for demographics merge and grouped validation only.
    subject = grouped[SUBJECT_COL].first().to_frame(SUBJECT_COL)
    pieces.append(subject)

    return pd.concat(pieces, axis=1).reset_index()

train_seq = make_sequence_features(train)
test_seq = make_sequence_features(test)

labels = train[[ID_COL, TARGET_COL]].drop_duplicates()
train_seq = train_seq.merge(labels, on=ID_COL, how="left")

print("train_seq:", train_seq.shape)
print("test_seq:", test_seq.shape)
display(train_seq.head())

Using sensor columns: ['acc_x', 'acc_y', 'acc_z', 'rot_w', 'rot_x', 'rot_y', 'rot_z']
train_seq: (6530, 60)
test_seq: (1621, 59)


,sequence_id,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_x_median,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_y_median,acc_z_mean,acc_z_std,acc_z_min,acc_z_max,acc_z_median,rot_w_mean,rot_w_std,rot_w_min,rot_w_max,rot_w_median,rot_x_mean,rot_x_std,rot_x_min,rot_x_max,rot_x_median,rot_y_mean,rot_y_std,rot_y_min,rot_y_max,rot_y_median,rot_z_mean,rot_z_std,rot_z_min,rot_z_max,rot_z_median,acc_x_first,acc_y_first,acc_z_first,rot_w_first,rot_x_first,rot_y_first,rot_z_first,acc_x_last,acc_y_last,acc_z_last,rot_w_last,rot_x_last,rot_y_last,rot_z_last,acc_x_delta,acc_y_delta,acc_z_delta,rot_w_delta,rot_x_delta,rot_y_delta,rot_z_delta,sequence_length,subject,gesture
0,SEQ_000007,6.153098,1.334155,3.613281,9.015625,6.488281,3.915570,3.048287,-2.019531,6.519531,5.488281,5.577782,2.337517,1.093750,9.792969,4.964844,0.263574,0.069033,0.134399,0.379272,0.254578,-0.280817,0.056597,-0.442871,-0.204163,-0.275757,-0.331470,0.175050,-0.478027,0.005066,-0.414978,-0.837994,0.040723,-0.914856,-0.757935,-0.825012,6.683594,6.214844,3.355469,0.134399,-0.355164,-0.447327,-0.809753,7.171875,5.984375,4.082031,0.243164,-0.264587,-0.413574,-0.836548,0.488281,-0.230469,0.726562,0.108765,0.090576,0.033752,-0.026794,57,SUBJ_059520,Cheek - pinch skin
1,SEQ_000008,3.400506,1.087142,1.734375,5.906250,3.437500,5.311179,3.268073,-0.222656,8.667969,7.000000,6.581629,2.475402,1.722656,11.074219,5.839844,0.243493,0.064414,0.157593,0.341980,0.226562,-0.117145,0.049384,-0.263306,-0.050537,-0.097382,-0.342327,0.190164,-0.508606,-0.031555,-0.442169,-0.875143,0.042626,-0.937805,-0.814697,-0.860046,2.765625,-0.222656,9.652344,0.319092,-0.132874,-0.031555,-0.937805,1.847656,4.718750,5.058594,0.305420,-0.160339,-0.093811,-0.933899,-0.917969,4.941406,-4.593750,-0.013672,-0.027466,-0.062256,0.003906,68,SUBJ_020948,Forehead - pull hairline
2,SEQ_000013,-7.058962,1.295184,-9.250000,-3.347656,-7.144531,2.346182,2.564639,-3.273438,4.683594,3.382812,-6.068544,1.330784,-10.945312,-3.515625,-5.851562,0.392208,0.150629,0.061157,0.540771,0.439514,0.340804,0.182002,0.140991,0.726501,0.258362,0.800506,0.090017,0.580505,0.881653,0.838135,0.002644,0.164305,-0.406799,0.129761,0.066101,-5.972656,3.160156,-6.453125,0.402161,0.405396,0.820801,0.014282,-5.992188,4.570312,-4.664062,0.510071,0.267151,0.811401,0.100342,-0.019531,1.410156,1.789062,0.107910,-0.138245,-0.009399,0.086060,53,SUBJ_040282,Cheek - pinch skin
3,SEQ_000022,8.020686,3.024375,0.777344,9.890625,9.746094,-0.503022,0.975726,-1.875000,2.492188,-0.800781,-2.407650,4.072718,-10.214844,0.933594,-0.175781,0.345264,0.191963,0.000916,0.477783,0.465515,-0.609041,0.267505,-0.901184,0.882019,-0.578674,-0.405867,0.151619,-0.605347,0.509094,-0.433838,-0.428959,0.175360,-0.570496,0.102051,-0.520569,9.742188,-1.644531,0.703125,0.446167,-0.608398,-0.326172,-0.569580,5.796875,2.492188,-7.257812,0.089539,-0.664062,-0.605347,-0.429626,-3.945312,4.136719,-7.960938,-0.356628,-0.055664,-0.279175,0.139954,159,SUBJ_024086,Feel around in tray and pull out an object
4,SEQ_000033,-4.506696,5.093991,-9.445312,7.562500,-7.130859,-1.683036,4.428661,-7.898438,10.023438,-2.324219,-0.815499,4.989123,-6.402344,10.492188,-2.781250,0.489975,0.173061,0.012695,0.819397,0.470245,-0.515358,0.360890,-0.776917,0.540222,-0.702972,0.224561,0.358961,-0.663879,0.661133,0.383789,0.096223,0.391796,-0.642761,0.597717,0.271545,-7.652344,-4.222656,-1.882812,0.012695,-0.453308,0.661133,0.597717,-4.851562,-7.132812,-2.800781,0.535400,-0.771423,0.252686,0.233154,2.800781,-2.910156,-0.917969,0.522705,-0.318115,-0.408447,-0.364563,56,SUBJ_040733,Neck - scratch


## 4. Merge demographics and select model features

Important: `subject` is an ID string.  
We keep it for grouped validation, but we **do not** feed it to the model.

To avoid string/imputer errors, the baseline uses numeric columns only.

In [5]:
train_seq = train_seq.merge(train_demo, on=SUBJECT_COL, how="left")
test_seq = test_seq.merge(test_demo, on=SUBJECT_COL, how="left")

# Save groups before removing subject from model features.
groups = train_seq[SUBJECT_COL].copy()

# Remove IDs and target from candidates.
excluded_cols = {ID_COL, TARGET_COL, SUBJECT_COL}
candidate_cols = [c for c in train_seq.columns if c not in excluded_cols]

# Baseline: numeric features only. This avoids object/string imputer errors.
feature_cols = [
    c for c in candidate_cols
    if pd.api.types.is_numeric_dtype(train_seq[c])
]

X = train_seq[feature_cols].copy()
X_test = test_seq[feature_cols].copy()
y_text = train_seq[TARGET_COL].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("Number of features:", len(feature_cols))
print("X:", X.shape)
print("X_test:", X_test.shape)
print("Number of classes:", len(label_encoder.classes_))
print(list(label_encoder.classes_))

# Sanity check: no object/string columns are passed to the model.
bad_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print("Non-numeric model columns:", bad_cols)
assert len(bad_cols) == 0

Number of features: 64
X: (6530, 64)
X_test: (1621, 64)
Number of classes: 18
['Above ear - pull hair', 'Cheek - pinch skin', 'Drink from bottle/cup', 'Eyebrow - pull hair', 'Eyelash - pull hair', 'Feel around in tray and pull out an object', 'Forehead - pull hairline', 'Forehead - scratch', 'Glasses on/off', 'Neck - pinch skin', 'Neck - scratch', 'Pinch knee/leg skin', 'Pull air toward your face', 'Scratch knee/leg skin', 'Text on phone', 'Wave hello', 'Write name in air', 'Write name on leg']
Non-numeric model columns: []


## 5. Train a baseline with cross-validation

Normal stratified CV is a simple starter.

For a stricter estimate, try the grouped CV section below, where validation subjects are unseen during training.

In [6]:
def make_model():
    if HAS_LGBM:
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=500,
                learning_rate=0.03,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=2026,
                objective="multiclass",
                verbosity=-1,
            ))
        ])
    else:
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                l2_regularization=0.05,
                random_state=2026,
            ))
        ])

N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=2026)

oof_pred = np.empty(len(y), dtype=object)
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    model = make_model()
    model.fit(X.iloc[train_idx], y[train_idx])

    pred_encoded = model.predict(X.iloc[valid_idx])
    pred_text = label_encoder.inverse_transform(pred_encoded.astype(int))
    true_text = y_text.iloc[valid_idx].to_numpy()

    score = hierarchical_f1_score(true_text, pred_text)
    fold_scores.append(score)
    oof_pred[valid_idx] = pred_text

    print(f"Fold {fold}: hierarchical F1 = {score:.5f}")

overall_score = hierarchical_f1_score(y_text, oof_pred)

print()
print("Fold scores:", [round(s, 5) for s in fold_scores])
print("Mean fold score:", np.mean(fold_scores))
print("OOF hierarchical F1:", overall_score)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1: hierarchical F1 = 0.67192


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2: hierarchical F1 = 0.67728


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 3: hierarchical F1 = 0.67620


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4: hierarchical F1 = 0.67634


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5: hierarchical F1 = 0.66728

Fold scores: [0.67192, 0.67728, 0.6762, 0.67634, 0.66728]
Mean fold score: 0.6738037849753371
OOF hierarchical F1: 0.6740556039128638


## 6. Optional stricter validation: grouped by subject

This better matches a subject-separated hidden test split.

It may score lower than normal CV. That is not necessarily bad; it may be more honest.

In [7]:
# Uncomment this section if you want stricter validation.

# cv_group = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2026)
# group_scores = []
#
# for fold, (train_idx, valid_idx) in enumerate(cv_group.split(X, y, groups), start=1):
#     model = make_model()
#     model.fit(X.iloc[train_idx], y[train_idx])
#     
#     pred_encoded = model.predict(X.iloc[valid_idx])
#     pred_text = label_encoder.inverse_transform(pred_encoded.astype(int))
#     true_text = y_text.iloc[valid_idx].to_numpy()
#     
#     score = hierarchical_f1_score(true_text, pred_text)
#     group_scores.append(score)
#     print(f"Group Fold {fold}: hierarchical F1 = {score:.5f}")
#
# print("Mean grouped score:", np.mean(group_scores))

## 7. Fit on all training data and create submission

Normal Kaggle submission format:

```csv
sequence_id,gesture
...
```

In [8]:
final_model = make_model()
final_model.fit(X, y)

test_pred_encoded = final_model.predict(X_test)
test_pred = label_encoder.inverse_transform(test_pred_encoded.astype(int))

pred_df = pd.DataFrame({
    ID_COL: test_seq[ID_COL],
    TARGET_COL: test_pred,
})

submission = sample_submission[[ID_COL]].merge(pred_df, on=ID_COL, how="left")

assert list(submission.columns) == [ID_COL, TARGET_COL]
assert len(submission) == len(sample_submission)
assert submission[TARGET_COL].isna().sum() == 0

submission.to_csv("submission.csv", index=False)

display(submission.head())
print("Saved submission.csv")
print(submission[TARGET_COL].value_counts())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,sequence_id,gesture
0,SEQ_000016,Write name on leg
1,SEQ_000018,Forehead - scratch
2,SEQ_000166,Forehead - pull hairline
3,SEQ_000169,Neck - pinch skin
4,SEQ_000174,Neck - scratch


Saved submission.csv
gesture
Forehead - scratch                            175
Text on phone                                 154
Neck - scratch                                149
Forehead - pull hairline                      144
Above ear - pull hair                         141
Neck - pinch skin                             141
Cheek - pinch skin                            116
Eyelash - pull hair                           104
Eyebrow - pull hair                            99
Wave hello                                     90
Write name in air                              78
Pull air toward your face                      72
Scratch knee/leg skin                          31
Drink from bottle/cup                          29
Feel around in tray and pull out an object     27
Glasses on/off                                 26
Pinch knee/leg skin                            25
Write name on leg                              20
Name: count, dtype: int64


## 8. Simple improvement ideas

Try:

- Add percentiles: 10%, 25%, 75%, 90%.
- Add early/middle/late window statistics.
- Add magnitude features, such as `sqrt(acc_x^2 + acc_y^2 + acc_z^2)`.
- Tune LightGBM parameters.
- Try ExtraTrees or RandomForest.
- Use grouped validation by subject to avoid overfitting to subject-specific patterns.